In [1]:
#Import Libs ( Crypto )
from cryptography.hazmat.primitives.keywrap import aes_key_wrap, aes_key_unwrap
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
from cryptography.hazmat.primitives.padding import PKCS7
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import hashes
import os

In [2]:
#Import Libs
import hashlib
import base64
import os

In [3]:
#File Upload.
path = "https://raw.githubusercontent.com/jrafa1607/Cryptography-In-Python/main/- Files/Test.txt"

In [4]:
#Files Path Definition
dec_path = path + ".dec"
enc_path = path + ".enc"

In [5]:
#Function to Derive Key
def derive_key(password: str, salt: bytes) -> bytes:

    kdf = PBKDF2HMAC(
        algorithm=hashes.SHA256(),
        length=32,
        salt=salt,
        iterations=100000,
        backend=default_backend()
    )
    return kdf.derive(password.encode())

In [6]:
# Function to Encrypt the Content in memory
def encrypt_file(plaintext_bytes: bytes, password: str) -> bytes:
    # Key + Salt
    salt = os.urandom(16)
    key = derive_key(password, salt)

    # Start Initialization Vector (IV)
    iv = os.urandom(16)

    # Cipher Setup
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv), backend=default_backend())
    encryptor = cipher.encryptor()

    # Padding Apply + AES Apply
    padder = PKCS7(algorithms.AES.block_size).padder()
    padded_data = padder.update(plaintext_bytes) + padder.finalize()
    ciphertext = encryptor.update(padded_data) + encryptor.finalize()

    # Return salt + iv + ciphertext as the encrypted content
    return salt + iv + ciphertext

In [7]:
# Function to Decrypt the Content in memory
def decrypt_file(encrypted_data_bytes: bytes, password: str) -> bytes:
    # SALT, IV + Cipher Data Extraction
    salt = encrypted_data_bytes[:16]
    iv = encrypted_data_bytes[16:32]
    ciphertext = encrypted_data_bytes[32:]

    # Key Derivation
    key = derive_key(password, salt)

    # Decrypt Setup
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv), backend=default_backend())
    decryptor = cipher.decryptor()

    # Padding Removing + Decrypt Apply
    padded_data = decryptor.update(ciphertext) + decryptor.finalize()
    unpadder = PKCS7(algorithms.AES.block_size).unpadder()
    plaintext = unpadder.update(padded_data) + unpadder.finalize()

    # Return plaintext bytes
    return plaintext

In [8]:
import requests # Needed to download files from URLs

# Download the file content directly into memory from the GitHub URL
print(f"Downloading file content from {path} into memory...")
try:
    response = requests.get(path)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    original_content_bytes = response.content # Store content as bytes

    print("File content downloaded successfully.")

    # Call the modified encrypt_file with the bytes content
    encrypted_content_bytes = encrypt_file(original_content_bytes, '1234')
    print("Content encrypted successfully.")

except requests.exceptions.RequestException as e:
    print(f"Error downloading the file: {e}")
except Exception as e:
    print(f"An unexpected error occurred during encryption: {e}")

File content downloaded successfully.
Content encrypted successfully.


In [9]:
# Decrypt Application using in-memory encrypted content
try:
    if 'encrypted_content_bytes' in locals() or 'encrypted_content_bytes' in globals():
        decrypted_content_bytes = decrypt_file(encrypted_content_bytes, '1234')
        print("Content decrypted successfully.")
    else:
        print("Error: encrypted_content_bytes not found. Please run the encryption cell first.")
except Exception as e:
    print(f"An unexpected error occurred during decryption: {e}")

Content decrypted successfully.


In [10]:
# Get the Values - Original File Content (from memory)
if 'original_content_bytes' in locals() or 'original_content_bytes' in globals():
    try:
        org = original_content_bytes.decode('utf-8')
        print("Original content loaded from memory.")
    except Exception as e:
        print(f"Error decoding original content from bytes: {e}")
else:
    print("Error: original_content_bytes not found. Please run the encryption cell first.")

Original content loaded from memory.


In [11]:
# Get the Values - Encrypted File Content (from memory, encoding with latin-1)
if 'encrypted_content_bytes' in locals() or 'encrypted_content_bytes' in globals():
    try:
        enc = encrypted_content_bytes.decode('latin-1') # Assuming encrypted bytes are best represented by latin-1 if non-UTF-8 characters are present
        print("Encrypted content loaded from memory.")
    except Exception as e:
        print(f"Error decoding encrypted content from bytes: {e}")
else:
    print("Error: encrypted_content_bytes not found. Please run the encryption cell first.")

Encrypted content loaded from memory.


In [12]:
# Get the Values - Decrypted File Content (from memory, encoding with utf-8)
if 'decrypted_content_bytes' in locals() or 'decrypted_content_bytes' in globals():
    try:
        dec = decrypted_content_bytes.decode('utf-8')
        print("Decrypted content loaded from memory.")
    except Exception as e:
        print(f"Error decoding decrypted content from bytes: {e}")
else:
    print("Error: decrypted_content_bytes not found. Please run the decryption cell first.")

Decrypted content loaded from memory.


In [13]:
#Show the Results
print("Original Content: " + org)
print("Encrypted Content: " + enc)
print("Decrypted Content: " + dec)

Original Content: Test Content

Encrypted Content: l`Üz~À¨FÆ®© N,A=/±¥5ß£øuê·ìã k«C¹¶(qé
Decrypted Content: Test Content

